# <font color="#76b900">**Notebook 2:** Evaluating Nemotron on the Cognitivo Financial Market Signal Task</font>

### Overview

The Cognitivo Hackathon asks teams to fine-tune **Llama-3.1-Nemotron-Nano-8B-v1** so it can synthesize grounded, concise answers to financial-market questions over the RBA cash-rate, ASX price, and AFR news datasets. The submitted agent architecture is:

```
question -> Qwen agent-brain plans and emits tool calls
         -> agent runtime executes query_data / retrieve
         -> tool results return to Qwen until reasoning is complete
         -> fine-tuned Nemotron synthesizes the final answer
         -> POST /query returns {"answer": "..."}
```

This notebook evaluates the **last step only** — whether Nemotron, given a question and the verified facts an upstream tool call would have produced, writes the direct, fact-complete `answer` the hackathon actually grades. It does not implement the Qwen planning loop or the `query_data`/`retrieve` tools themselves (that belongs in `src/`); it isolates the domain-model synthesis step so it can be measured and improved independently, both before and after fine-tuning.

We use the **[NeMo Evaluator SDK](https://github.com/NVIDIA-NeMo/Evaluator)** (as in the original workshop template) against the 15 labeled practice cases in `Participant_Package/public_questions.jsonl`, and measure two things:

1. **Similarity metrics (BLEU / ROUGE / F1)** against each question's reference answer — a cheap surface-level signal.
2. **A component-based LLM judge** that reproduces the hackathon's own grading rubric — checking, per question, whether every `grading.components[].expected_fact` is present in the model's answer, and summing the attached `points`. This is the metric that actually predicts hidden-question performance.

### Learning Objectives

In this notebook you will learn how to:
- Drive the NeMo Evaluator SDK against a locally-served Nemotron NIM/endpoint
- Register the hackathon's public questions as a **bring-your-own-benchmark (BYOB)** evaluation
- Compare an **ungrounded** answer (question only — the "no tool use" failure mode the Challenge Brief calls out) against a **grounded** answer (question + verified facts, i.e. what Nemotron actually receives from the agent runtime)
- Build an LLM-judge rubric that mirrors the official **component-based, partial-credit** grading in `validate.json` / `public_questions.jsonl`
- Re-run the same notebook against the fine-tuned model endpoint to produce the base-vs-fine-tuned comparison required for the **Fine-Tuned Model Quality (30%)** scoring pillar

> **Prerequisites:** a Nemotron NIM or OpenAI-compatible endpoint serving `nvidia/llama-3.1-nemotron-nano-8b-v1` (see [`00-NIM-Setup-DGX-Spark.ipynb`](00-NIM-Setup-DGX-Spark.ipynb) and the Participant Package's [Setup Instructions](../../Participant_Package/Setup_Instructions.md)), and the NeMo Evaluator SDK installed per [`00b-NeMo-Evaluator-SDK-Setup-DGX-Spark.ipynb`](00b-NeMo-Evaluator-SDK-Setup-DGX-Spark.ipynb).

### Why "ungrounded vs. grounded" instead of zero-shot vs. few-shot ICL?

The original legal-title task compared zero-shot prompting to few-shot in-context learning. The financial-answer-synthesis task has a more relevant axis, taken directly from the Challenge Brief's own **Required Model Roles** and **"What zero looks like"** sections:

- **Ungrounded**: the model receives only the question, with no tool results — the same failure mode as the Brief's *"No tool use"* zero-score example, where the model "thinks out loud" or invents a plausible-sounding but ungrounded answer.
- **Grounded**: the model receives the question **plus the verified facts an upstream `query_data`/`retrieve` call would have returned** — exactly what the Brief specifies Nemotron receives in the real pipeline: *"the question and accumulated verified tool results after the Qwen reasoning loop."*

Comparing these two conditions quantifies how much of the answer quality comes from grounding versus from the model itself — and, once you fine-tune, how much fine-tuning improves faithful synthesis **given** grounding (rather than teaching the model to memorize facts it should instead be reading from tool output).

<br><hr>

### How evaluation works on the Spark

The NeMo Evaluator SDK (`nemo-evaluator`, the `nel` CLI) requires **Python 3.12+**.
The `llm-eval-workshop` conda env is created at Python 3.12, so the SDK is
installed **in the same environment as this kernel** — no separate env, no conda
switching.

`_nel()` in the config cell below calls the `nel` binary that lives next to this
kernel's Python interpreter.

One model endpoint does everything in this notebook: the same served Nemotron model is used both as the **model under test** and as the **LLM judge**. This is a convenience for local iteration — see the self-judge caveat later in this notebook.

### Configuration

This is the **single place** to change the endpoint, model, or evaluator env. Every cell below reads these values and the helper functions defined here.

Point `NIM_HOST`/`MODEL_ID` at whichever endpoint you want to score: the base model now, and the fine-tuned `domain-ft` endpoint (see the Participant Package's Setup Instructions — port `8001` behind the `domain-ft` alias) once your adapter is trained and served. Re-running this notebook against both endpoints is how you produce the base-vs-fine-tuned evidence required for scoring.

In [ ]:
import os, sys, subprocess, json, glob, shutil
from pathlib import Path
import requests

# === DGX Spark configuration =================================================
NIM_HOST = "http://localhost:8000"
MODEL_ID = "nvidia/llama-3.1-nemotron-nano-8b-v1"
CHAT_URL = f"{NIM_HOST}/v1/chat/completions"
DATA_DIR = "data"
QUESTIONS_PATH = Path("../../Participant_Package/public_questions.jsonl")

os.makedirs("nel_benchmarks", exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
_NEL = str(Path(sys.executable).parent / "nel")
if not Path(_NEL).is_file():
    raise FileNotFoundError(
        f"NeMo Evaluator CLI not found at {_NEL}. Run ./setup-dgx-spark.sh first."
    )

def _nel(args):
    """Run the nel CLI from the active environment, streaming output live."""
    cmd = [_NEL] + args
    print("\u2192", " ".join(cmd), "\n")
    subprocess.run(cmd, check=True)

def run_bench(bench, out_dir, max_problems=None, max_tokens=256, system_prompt=None):
    """Run one benchmark against the local model endpoint."""
    shutil.rmtree(out_dir, ignore_errors=True)
    args = ["eval", "run", "--bench", bench,
            "--model-url", CHAT_URL, "--model-id", MODEL_ID, "--api-key", "dummy",
            "--max-tokens", str(max_tokens), "--output-dir", out_dir]
    if max_problems is not None:
        args += ["--max-problems", str(max_problems)]
    if system_prompt:
        args += ["--system-prompt", system_prompt]
    _nel(args)
    return out_dir

def run_config(cfg_path):
    """Run a full YAML config (used for LLM-as-a-judge)."""
    _nel(["eval", "run", cfg_path])

def load_results(out_dir):
    """Load per-sample records from results.jsonl."""
    files = glob.glob(os.path.join(out_dir, "**", "results.jsonl"), recursive=True)
    if not files:
        raise FileNotFoundError(f"No results.jsonl under {out_dir}")
    with open(files[0]) as f:
        return [json.loads(line) for line in f]

def load_scores(out_dir):
    """Load aggregate scores from the eval report."""
    files = glob.glob(os.path.join(out_dir, "**", "eval-*.json"), recursive=True)
    if not files:
        raise FileNotFoundError(f"No eval report under {out_dir}")
    with open(files[0]) as f:
        return json.load(f)["benchmark"]["scores"]

def mean_metrics(rows, keys):
    """Mean of named scoring_details fields across samples."""
    out = {}
    for key in keys:
        values = [row["scoring_details"].get(key) for row in rows
                  if isinstance(row.get("scoring_details"), dict)
                  and isinstance(row["scoring_details"].get(key), (int, float))]
        out[key] = sum(values) / len(values) if values else float("nan")
    return out

os.environ.update(NIM_HOST=NIM_HOST, MODEL_ID=MODEL_ID)
print("NIM_HOST =", NIM_HOST, "| MODEL_ID =", MODEL_ID)
print("data dir =", DATA_DIR)
print("questions =", QUESTIONS_PATH, "(exists:", QUESTIONS_PATH.is_file(), ")")
print("nel      =", _NEL)

Verify the model endpoint is serving and the NeMo Evaluator SDK is available in the active `llm-eval-workshop` environment:

In [ ]:
models_response = requests.get(f"{NIM_HOST}/v1/models", timeout=5)
models_response.raise_for_status()
served_model = models_response.json()["data"][0]["id"]
print("Serving model:", served_model)
assert served_model == MODEL_ID, f"Expected {MODEL_ID}, but the endpoint serves {served_model}."
_nel(["--version"])

### Load the public calibration questions

`Participant_Package/public_questions.jsonl` holds the **15 public practice cases** in the same
schema the hidden evaluation harness uses. Each line is one JSON object with `id`, `prompt`,
`difficulty`, `datasets`, `dataset_scope`, a `reference_answer`, and a `grading` object listing the
`expected_fact` / `points` pairs the judge checks — this is exactly what
[`Setup_Instructions.md`](../../Participant_Package/Setup_Instructions.md) and the Challenge Brief
describe as the grading contract.

There is no larger held-out split to pull additional examples from — these 15 cases are the only
labeled data participants get, so we evaluate on all of them rather than holding any out.

> **Scope note:** this notebook does not implement question-ID-specific hard-coded answers, and it
> does not re-derive the RBA/ASX/AFR facts itself — the `grading.components` already encode the
> correct facts for these public cases. It only tests whether the domain model, given those facts
> as if an upstream tool call had already verified them, writes the answer the grader is actually
> looking for.

In [ ]:
from pprint import pp
import json
import pandas as pd

In [ ]:
def read_questions(path):
    with path.open() as f:
        return [json.loads(line) for line in f]

questions = read_questions(QUESTIONS_PATH)
print(f"Loaded {len(questions)} public questions")

overview = pd.DataFrame([
    {"id": q["id"], "difficulty": q["difficulty"], "datasets": ",".join(q["datasets"]),
     "scope": q["dataset_scope"], "components": len(q["grading"]["components"])}
    for q in questions
])
print(overview.to_string(index=False))
pp(questions[0])

#### Build the "grounded" context (verified tool-result facts)

For the **grounded** condition we build a text block that stands in for the agent runtime's
`tool_trace` results — the facts Nemotron would actually receive after Qwen's tool-calling loop
finishes. We take them straight from each question's `grading.components`, since those already are
the verified facts the judge checks for; we present them as an unordered list of facts rather than
as ready-made prose, so the model still has to do real synthesis (selecting, ordering, and phrasing
them into the required direct-answer format) rather than simply copying a sentence.

For the **ungrounded** condition we give the model nothing but the question — reproducing the
Brief's *"no tool use"* failure mode so we can measure how much the grounding step is worth.

In [ ]:
def build_grounding_context(q):
    lines = ["Verified facts returned by upstream data-query/retrieval tools:"]
    for c in q["grading"]["components"]:
        lines.append(f"- {c['expected_fact']}")
    tolerance = q["grading"].get("tolerance_note")
    if tolerance:
        lines.append(f"(Tolerance: {tolerance})")
    return "\n".join(lines)

def make_prompt(q, grounded):
    if grounded:
        return f"{build_grounding_context(q)}\n\nQuestion: {q['prompt']}\nAnswer:"
    return f"Question: {q['prompt']}\nAnswer:"

print("--- ungrounded ---")
print(make_prompt(questions[0], grounded=False))
print("\n--- grounded ---")
print(make_prompt(questions[0], grounded=True))

#### Create the ungrounded and grounded evaluation files

Each BYOB benchmark reads a local JSONL file with a `prompt` and a reference `completion`. We write
**four** files into `data/`, two conditions times two scorer types:

- `fin-qa-eval-{ungrounded,grounded}.jsonl` — `completion` is the plain `reference_answer`, read by
  the **similarity** scorer (BLEU/ROUGE/F1).
- `fin-qa-judge-{ungrounded,grounded}.jsonl` — `completion` is a JSON-encoded grading spec
  (`expected_facts`, `max_score`, `tolerance_note`), read by the **component-based judge** rubric
  defined later.

We also keep a `PROMPT_META` lookup so we can join judge/similarity results back to each question's
`id` and `difficulty` for a breakdown table, since the exact prompt text we write here is echoed
back verbatim in `results.jsonl`.

In [ ]:
def write_eval_files(qs):
    sim_paths = {False: os.path.join(DATA_DIR, "fin-qa-eval-ungrounded.jsonl"),
                 True:  os.path.join(DATA_DIR, "fin-qa-eval-grounded.jsonl")}
    judge_paths = {False: os.path.join(DATA_DIR, "fin-qa-judge-ungrounded.jsonl"),
                   True:  os.path.join(DATA_DIR, "fin-qa-judge-grounded.jsonl")}
    prompt_meta = {}

    for grounded in (False, True):
        with open(sim_paths[grounded], "w") as fs, open(judge_paths[grounded], "w") as fj:
            for q in qs:
                prompt = make_prompt(q, grounded)
                fs.write(json.dumps({
                    "prompt": prompt, "completion": q["reference_answer"],
                    "category": "summarization",
                }) + "\n")
                grading_spec = {
                    "expected_facts": [
                        {"expected_fact": c["expected_fact"], "points": c["points"]}
                        for c in q["grading"]["components"]
                    ],
                    "max_score": q["grading"]["max_score"],
                    "tolerance_note": q["grading"].get("tolerance_note", ""),
                }
                fj.write(json.dumps({
                    "prompt": prompt, "completion": json.dumps(grading_spec),
                    "category": "summarization",
                }) + "\n")
                prompt_meta[prompt] = {
                    "id": q["id"], "difficulty": q["difficulty"],
                    "datasets": ",".join(q["datasets"]), "grounded": grounded,
                }
    written = list(sim_paths.values()) + list(judge_paths.values())
    print(f"Wrote {len(qs)} rows x 2 conditions to {written}")
    return prompt_meta

PROMPT_META = write_eval_files(questions)

<br><hr>

## Evaluation with the NeMo Evaluator SDK

As in the original template, we evaluate our own data with a **bring-your-own-benchmark (BYOB)**
module: a small `.py` file that declares the dataset, prompt template, and reference field, plus a
`@scorer` function. We define four benchmarks that share two scorers, one per condition per
scorer type.

#### Evaluation using similarity metrics

Similarity metrics compare the model's answer against the `reference_answer` text. They're a cheap,
fast sanity signal, but for this task they're a weak proxy for correctness: two answers can share
most of their words and still get the ticker, the sign, or the date wrong, which the hackathon
grades as zero for that component. Treat this section as a quick smoke test; the component-based
judge below is the metric that matters.

| Metric | Measures |
| --- | --- |
| **BLEU** | n-gram overlap, precision-weighted |
| **ROUGE-1** | unigram overlap |
| **ROUGE-L** | longest common subsequence |
| **F1** | harmonic mean of token precision & recall |

In [ ]:
%%writefile nel_benchmarks/fin_qa_similarity.py
"""BYOB financial-answer benchmarks (ungrounded & grounded) with BLEU/ROUGE/F1 scoring."""
from nemo_evaluator.environments.custom import benchmark, scorer
from nemo_evaluator.scoring import ScorerInput
from rouge_score import rouge_scorer
import sacrebleu

_rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

# Llama-3.1-Nemotron-Nano-8B-v1 toggles its reasoning trace with a "detailed thinking
# on/off" system-prompt convention (different from the "/no_think" convention used by
# nemotron-nano-9b-v2 in the original template) -- check the model card if your build
# behaves differently. We turn it off so the reply is the final answer, not a trace.
SYSTEM = ("detailed thinking off\n"
          "You are the domain answer-synthesis model in a financial-market agent. Given a "
          "question, and any verified facts supplied above it, write ONE direct answer that "
          "states every value the question asks for, using only the supplied facts. Do not "
          "invent figures, do not describe your reasoning, and do not hedge -- output only "
          "the final answer.")

@scorer
def fin_scorer(sample: ScorerInput) -> dict:
    ref = str(sample.target).strip()
    hyp = sample.response.strip().split("\n")[0].strip().strip('"')
    r = _rs.score(ref, hyp)
    bleu = sacrebleu.sentence_bleu(hyp, [ref]).score / 100.0
    rt, ht = set(ref.lower().split()), set(hyp.lower().split())
    inter = len(rt & ht)
    prec = inter / len(ht) if ht else 0.0
    rec = inter / len(rt) if rt else 0.0
    f1 = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
    return {"reward": r["rougeL"].fmeasure, "bleu": bleu,
            "rouge1": r["rouge1"].fmeasure, "rougeL": r["rougeL"].fmeasure, "f1": f1}

benchmark(name="fin-qa-ungrounded", dataset="data/fin-qa-eval-ungrounded.jsonl", prompt="{prompt}",
          target_field="completion", system_prompt=SYSTEM)(fin_scorer)
benchmark(name="fin-qa-grounded", dataset="data/fin-qa-eval-grounded.jsonl", prompt="{prompt}",
          target_field="completion", system_prompt=SYSTEM)(fin_scorer)


#### Run the ungrounded similarity evaluation

In [ ]:
run_bench("nel_benchmarks/fin_qa_similarity.py:fin-qa-ungrounded",
          "results/sim_ungrounded", max_problems=len(questions), max_tokens=256)

#### Run the grounded similarity evaluation

In [ ]:
run_bench("nel_benchmarks/fin_qa_similarity.py:fin-qa-grounded",
          "results/sim_grounded", max_problems=len(questions), max_tokens=256)

#### Compare ungrounded vs grounded

In [ ]:
keys = ["bleu", "rouge1", "rougeL", "f1"]
ungrounded = mean_metrics(load_results("results/sim_ungrounded"), keys)
grounded  = mean_metrics(load_results("results/sim_grounded"),  keys)

df = pd.DataFrame({"ungrounded": ungrounded, "grounded": grounded})
df["delta"] = df["grounded"] - df["ungrounded"]
print(df.round(4))

A large positive `delta` here mostly confirms that giving the model the verified facts helps it
say the right words — expected, since we handed it half the reference answer's vocabulary as
"facts." It does **not** tell you whether the model got the *values* right (a wrong ticker or a
transposed sign can still score well on n-gram overlap). That's what the judge below checks.

<br><hr>

### Evaluation with LLM-as-a-Judge

Instead of scoring word overlap, we ask a judge model to check the candidate answer against the
**same component-based rubric the hackathon itself uses**: for every `expected_fact` in
`grading.components`, does the candidate answer state it (allowing the tolerances and equivalent
phrasing the Brief permits)? The judge sums the `points` of every satisfied fact into a score out
of `max_score` — reproducing, on the public set, exactly the arithmetic that scores the hidden
questions.

> **On the Spark** we reuse the **same local model as the judge** (fully offline, no API key). A
> model judging its own family is less reliable than a larger independent judge, so we set
> `allow_self_judge: true` and treat these scores as directional. For anything you report as
> evidence, also spot-check a handful of `reasoning` fields by hand, and consider pointing `judge`
> at the Qwen `agent-brain` endpoint instead if it's available to you locally.

#### Define the judge benchmark and evaluation config

The judge benchmark's scorer returns `needs_judge(...)`, deferring to the judge model. The config
declares two `api` services — a **solver** (the model under test) and a **judge** — plus a `judge`
metric whose rubric is built from each question's grading spec rather than a fixed 1-5 rubric.

In [ ]:
%%writefile nel_benchmarks/fin_qa_judge.py
"""BYOB financial-answer benchmarks that defer scoring to a component-based LLM judge."""
from nemo_evaluator.environments.custom import benchmark, scorer
from nemo_evaluator.scoring import ScorerInput, needs_judge

SYSTEM = ("detailed thinking off\n"
          "You are the domain answer-synthesis model in a financial-market agent. Given a "
          "question, and any verified facts supplied above it, write ONE direct answer that "
          "states every value the question asks for, using only the supplied facts. Do not "
          "invent figures, do not describe your reasoning, and do not hedge -- output only "
          "the final answer.")

@scorer
def judge_defer(sample: ScorerInput) -> dict:
    # Returning needs_judge(...) tells the eval loop to score this sample with the LLM judge
    # configured in the eval config (see make_judge_config below).
    return needs_judge(sample)

for _name, _ds in [("fin-qa-judge-ungrounded", "data/fin-qa-judge-ungrounded.jsonl"),
                   ("fin-qa-judge-grounded", "data/fin-qa-judge-grounded.jsonl")]:
    benchmark(name=_name, dataset=_ds, prompt="{prompt}",
              target_field="completion", system_prompt=SYSTEM)(judge_defer)


In [ ]:
import yaml

RUBRIC = (
    "You are grading a candidate ANSWER against the official Cognitivo Hackathon "
    "component-based rubric.\n{instruction}\n"
    "Candidate answer to grade:\n{response}\n{reference_section}\n"
    "The reference above is a JSON object with an `expected_facts` list (each entry has an "
    "`expected_fact` and its `points`), a `max_score`, and a `tolerance_note`.\n"
    "For every entry in `expected_facts`, decide whether the candidate answer states that fact. "
    "Accept equivalent date formats, harmless numeric formatting differences, and synonyms that "
    "preserve meaning, honoring `tolerance_note`. A fact fails if it is missing, contradicted, or "
    "replaced with an invented value.\n"
    "Sum the `points` of every satisfied fact into a single total out of `max_score`.\n"
    'Reply with only a JSON object: {{"score": <total out of max_score>, '
    '"reasoning": "<which facts matched or failed, briefly>"}}'
)

def make_judge_config(bench, out_dir, max_problems=None):
    svc = {"type": "api", "url": CHAT_URL, "protocol": "chat_completions",
           "model": MODEL_ID, "api_key": "dummy"}
    cfg = {
        "services": {"solver": svc, "judge": dict(svc)},
        "sandboxes": {"none": {"type": "none"}},
        "benchmarks": [{
            "name": f"nel_benchmarks/fin_qa_judge.py:{bench}",
            "solver": {"type": "simple", "service": "solver", "system_prompt": "detailed thinking off"},
            "max_problems": max_problems if max_problems is not None else len(questions),
            "sandbox": {"type": "none"},
            "scoring": {"include_defaults": True, "metrics": [{
                "type": "judge", "name": "quality", "service": "judge",
                "reference_free": False, "allow_self_judge": True, "max_score": 10,
                "system_prompt": "detailed thinking off\nReply only with the JSON object.",
                "rubric": RUBRIC}]},
            "params": {}}],
        "cluster": {"type": "local"},
        "output": {"dir": out_dir, "timestamped": False},
    }
    path = f"nel_benchmarks/{bench}.yaml"
    yaml.safe_dump(cfg, open(path, "w"), sort_keys=False)
    return path

#### Run the judge evaluation (ungrounded)

The judge runs with reasoning on by default for the judge call, so this is slower than the
similarity runs — 15 samples takes a few minutes.

In [ ]:
cfg_ungrounded = make_judge_config("fin-qa-judge-ungrounded", "results/judge_ungrounded")
run_config(cfg_ungrounded)

#### Inspect the ungrounded judge scores

Each sample gets a judge `score` (out of `max_score`) and a `normalized` value (0-1), plus the
judge's reasoning about which facts matched.

In [ ]:
def judge_score_table(out_dir, condition):
    rows = load_results(out_dir)
    recs = []
    for r in rows:
        meta = PROMPT_META.get(r["prompt"], {})
        j = (r.get("scoring_details") or {}).get("judge") or {}
        recs.append({"id": meta.get("id"), "difficulty": meta.get("difficulty"),
                     "condition": condition, "score": j.get("score"),
                     "normalized": j.get("normalized")})
    return pd.DataFrame(recs)

ungrounded_scores = judge_score_table("results/judge_ungrounded", "ungrounded")
print(f"Ungrounded judge mean: {ungrounded_scores['normalized'].mean():.3f}  "
      f"(n={len(ungrounded_scores)})\n")
print(ungrounded_scores.to_string(index=False))

rows = load_results("results/judge_ungrounded")
sample = rows[0]
print("\nQuestion:", sample["prompt"].split("Question:")[-1][:120].strip(), "...")
print("Model answer :", sample["model_response"].strip().splitlines()[0])
print("Judge score  :", sample["scoring_details"]["judge"]["score"],
      "/10 ->", round(sample["scoring_details"]["judge"]["normalized"], 2))
print("Judge says   :", sample["scoring_details"]["judge"]["reasoning"][:200], "...")

#### Run the judge evaluation (grounded)

Same flow, pointed at the grounded condition — this tells you what Nemotron's synthesis quality
looks like when it actually gets the tool results it's supposed to get in the real pipeline.

In [ ]:
cfg_grounded = make_judge_config("fin-qa-judge-grounded", "results/judge_grounded")
run_config(cfg_grounded)
grounded_scores = judge_score_table("results/judge_grounded", "grounded")
print(f"Grounded judge mean: {grounded_scores['normalized'].mean():.3f}  "
      f"(n={len(grounded_scores)})")

#### Compare ungrounded vs grounded, overall and by difficulty

In [ ]:
all_scores = pd.concat([ungrounded_scores, grounded_scores], ignore_index=True)

overall = all_scores.groupby("condition")["normalized"].mean().rename("mean_normalized")
print("Overall:\n", overall.round(3), "\n")

by_difficulty = all_scores.pivot_table(index="difficulty", columns="condition",
                                        values="normalized", aggfunc="mean")
by_difficulty["delta"] = by_difficulty.get("grounded") - by_difficulty.get("ungrounded")
print("By difficulty:\n", by_difficulty.round(3))

<br><hr>

### Wrap-up

You measured Nemotron's financial-answer **synthesis** quality two ways — cheap similarity metrics
and a component-based LLM judge that mirrors the hackathon's own grading — under an **ungrounded**
condition (the Brief's "no tool use" failure mode) and a **grounded** condition (question + verified
tool facts, matching what the real agent runtime hands to `DOMAIN_FT_MODEL`).

Next steps for the hackathon submission:

1. **Fine-tune** `nvidia/llama-3.1-nemotron-nano-8b-v1` on domain training examples (see the
   reference LoRA configuration in [`Setup_Instructions.md`](../../Participant_Package/Setup_Instructions.md)).
2. Serve the resulting adapter behind the `domain-ft` alias (port `8001`), then re-run this
   notebook with `NIM_HOST`/`MODEL_ID` pointed at that endpoint.
3. Save the resulting `results/` output and the ungrounded-vs-grounded / base-vs-fine-tuned
   comparison tables into `training/` as the base-vs-fine-tuned evidence the **Fine-Tuned Model
   Quality (30%)** pillar requires.
4. Wire this same domain model into the real agent (`src/`) behind `DOMAIN_FT_MODEL`, switch
   `DOMAIN_PREDICT_MODE` from `mock` to `llm`, and confirm `POST /query` on a few public questions
   before relying on the hidden-question harness.